In [1]:
##### SETUP
from random import shuffle
import adalflow as adal
from typing import Dict
from adalflow.optim.types import ParameterType
from adalflow.components.model_client.openai_client import OpenAIClient
from common import (compute_spec_score, 
                    clean_output, 
                    set_seed, 
                    load_data,
                    load_data_c1,
                    load_data_c2, 
                    BASELINE_PROMPT,
                    EVAL_FN_DESCRIPTION, 
                    BATCH_SIZE, 
                    MAX_STEPS, 
                    NUM_WORKERS)

In [2]:
template = r"""<START_OF_SYSTEM_PROMPT>
{{system_prompt}}
<END_OF_SYSTEM_PROMPT>
<START_OF_USER>
{{input_str}}
<END_OF_USER>
"""

#template = r"""<START_OF_SYSTEM_PROMPT>
#{{system_prompt}}
#<END_OF_SYSTEM_PROMPT>
#<START_OF_USER_PROMPT>
#{{input_str}}
#<END_OF_USER_PROMPT>
#"""


class AutonomousDrivingTaskPipeline(adal.Component):
    def __init__(self, model_client: adal.ModelClient, model_kwargs: Dict):
        super().__init__()

        system_prompt = adal.Parameter(
            data=BASELINE_PROMPT,
            role_desc="To give task instruction to the language model in the system prompt",
            requires_opt=True,
            param_type=ParameterType.PROMPT,
        )

        self.llm_driver = adal.Generator(
            model_client=model_client,
            model_kwargs=model_kwargs,
            template=template,
            prompt_kwargs={
                "system_prompt": system_prompt,
            },
            output_processors=clean_output,
            use_cache=True,
        )
    
    def call(self, question: str, id: str = None):
        return self.llm_driver(prompt_kwargs={"input_str": question}, id=id)

In [3]:
gpt_4o_model = {
    "model_client": OpenAIClient(),
    "model_kwargs": {
        "model": "gpt-4o",
        "max_tokens": 4000,
        "temperature": 0.0,
        "top_p": 0.99,
        "frequency_penalty": 0,
        "presence_penalty": 0,
        "stop": None,
    },
}



In [4]:
from adalflow.core import DataClass
from dataclasses import dataclass, field

@dataclass
class ADData(DataClass):
    question: str = field(
        metadata={"desc": "Driving task"}
    )
    id: int = field(
        metadata={"desc": "Task id"}
    )

In [5]:
class AutonomousDrivingAdalComponent(adal.AdalComponent):  # noqa: F811
    def __init__(
        self,
        model_client: adal.ModelClient,
        model_kwargs: Dict,
        backward_engine_model_config: Dict,
        #teacher_model_config: Dict,
        text_optimizer_model_config: Dict,
    ):
        task = AutonomousDrivingTaskPipeline(model_client, model_kwargs)
        eval_fn = compute_spec_score
        loss_fn = adal.EvalFnToTextLoss(
            eval_fn=eval_fn,
            eval_fn_desc=EVAL_FN_DESCRIPTION,
        )
        super().__init__(task=task, eval_fn=eval_fn, loss_fn=loss_fn)

        self.backward_engine_model_config = backward_engine_model_config
        #self.teacher_model_config = teacher_model_config
        self.text_optimizer_model_config = text_optimizer_model_config
    
    def prepare_task(self, sample: ADData):
        return self.task.call, {"question": sample.question, "id": sample.id}

    def prepare_eval(self, sample: ADData, y_pred: adal.GeneratorOutput) -> float:
        return self.eval_fn, {"pred": y_pred.data}

    def prepare_loss(self, sample: ADData, pred: adal.Parameter):
        pred.eval_input = pred.full_response.data
        return self.loss_fn, {"kwargs": {"pred": pred}}

In [6]:
def train(
    train_batch_size=3,  
    max_steps=12,
    strategy="random",
    optimization_order="sequential",
    debug=False,
    resume_from_ckpt=None,
    data_c=0
):
    adal_component = AutonomousDrivingAdalComponent(
        **gpt_4o_model, # Changed from 3 
        #teacher_model_config=gpt_4o_model_opt,
        text_optimizer_model_config=gpt_4o_model,
        backward_engine_model_config=gpt_4o_model,
    )

    trainer = adal.Trainer(
        train_batch_size=train_batch_size,
        adaltask=adal_component,
        strategy=strategy,
        max_steps=max_steps,
        num_workers=NUM_WORKERS,
        debug=debug,
        weighted_sampling=True,
        optimization_order=optimization_order,
    )
    
    if data_c == 1:
        train_set, val_set, test_set = load_data_c1() 
    elif data_c ==2:
        train_set, val_set, test_set = load_data_c2() 
    else:
        train_set, val_set, test_set = load_data() 
    train_set = [ADData(q, i) for i,q in  enumerate(train_set)]
    val_set = [ADData(q, i+20) for i,q in  enumerate(val_set)]
    test_set = [ADData(q, i+40) for i,q in  enumerate(test_set)]
    #train_set = train_set * 10
    #shuffle(train_set)

    new_ts = []
    for _ in range(10):
        shuffle(train_set)
        new_ts += train_set

    train_set = new_ts

    trainer.fit(
        train_dataset=train_set,
        val_dataset=val_set,
        test_dataset=test_set,
        debug=debug,
        resume_from_ckpt=resume_from_ckpt,
    )


In [ ]:
#%%capture cap
train(
    train_batch_size = BATCH_SIZE,
    debug=False,
    max_steps=MAX_STEPS,
    strategy="random",
    data_c=1
    #resume_from_ckpt="/Users/gjperin/.adalflow/ckpt/AutonomousDrivingAdalComponent/random_max_steps_5_3e373_run_2.json"
)
#with open('output.txt', 'w') as file:
#    file.write(cap.stdout)


In [ ]:
train(
    train_batch_size = BATCH_SIZE,
    debug=False,
    max_steps=MAX_STEPS,
    strategy="random",
    data_c=1
    #resume_from_ckpt="/Users/gjperin/.adalflow/ckpt/AutonomousDrivingAdalComponent/random_max_steps_5_3e373_run_2.json"
)

In [ ]:
train(
    train_batch_size = BATCH_SIZE,
    debug=False,
    max_steps=MAX_STEPS,
    strategy="random",
    data_c=1
    #resume_from_ckpt="/Users/gjperin/.adalflow/ckpt/AutonomousDrivingAdalComponent/random_max_steps_5_3e373_run_2.json"
)

In [ ]:
train(
    train_batch_size = BATCH_SIZE,
    debug=False,
    max_steps=MAX_STEPS,
    strategy="random",
    data_c=2
    #resume_from_ckpt="/Users/gjperin/.adalflow/ckpt/AutonomousDrivingAdalComponent/random_max_steps_5_3e373_run_2.json"
)

In [ ]:
train(
    train_batch_size = BATCH_SIZE,
    debug=False,
    max_steps=MAX_STEPS,
    strategy="random",
    data_c=2
    #resume_from_ckpt="/Users/gjperin/.adalflow/ckpt/AutonomousDrivingAdalComponent/random_max_steps_5_3e373_run_2.json"
)

In [ ]:
train(
    train_batch_size = BATCH_SIZE,
    debug=False,
    max_steps=MAX_STEPS,
    strategy="random",
    data_c=2
    #resume_from_ckpt="/Users/gjperin/.adalflow/ckpt/AutonomousDrivingAdalComponent/random_max_steps_5_3e373_run_2.json"
)